In [56]:
!pip install requests beautifulsoup4 pandas -q
!pip install lxml -q
!pip install html5lib -q
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q

In [57]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [58]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [59]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import os
import re

# ─── CONFIG ───────────────────────────────────────────────
API_KEY    =  G_SCRAPER_API_KEY
TARGET_URL = "https://www.klsescreener.com/v2/screener/quote_results"
OUTPUT_DIR = "output_tables"
RENDER_JS  = False  # Set True if tables are loaded via JavaScript
# ──────────────────────────────────────────────────────────

def sanitize_filename(name):
    """Remove characters that are invalid in filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).strip().replace(" ", "_")

def fetch_page(url):
    params = {
        "api_key": API_KEY,
        "url": url,
    }
    if RENDER_JS:
        params["render"] = "true"

    print(f"Fetching: {url}")
    response = requests.get("http://api.scraperapi.com", params=params)
    response.raise_for_status()
    return response.text

def extract_table_names(html):
    """Try to get a meaningful name from <caption> or nearest heading above each table."""
    soup = BeautifulSoup(html, "html.parser")
    raw_tables = soup.find_all("table")
    names = []

    for i, table in enumerate(raw_tables):
        # Try <caption> tag first
        caption = table.find("caption")
        if caption:
            names.append(sanitize_filename(caption.text))
            continue

        # Try nearest preceding sibling heading (h1–h4)
        heading = None
        for sibling in table.find_all_previous(["h1", "h2", "h3", "h4"]):
            heading = sibling.text.strip()
            break

        if heading:
            names.append(sanitize_filename(heading))
        else:
            names.append(f"table_{i + 1}")

    return names



def scrape_tables_to_csv(url):
    html = fetch_page(url)

    # Parse all tables
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        print("No tables found on the page.")
        return

    print(f"Found {len(tables)} table(s)")

    # Get table names
    names = extract_table_names(html)

    # Ensure names list matches number of tables
    while len(names) < len(tables):
        names.append(f"table_{len(names) + 1}")

    # Save each table
    #for i, (df, name) in enumerate(zip(tables, names)):
    df = tables[0]
    return df
    #df.info()
    #print(df.head())

if __name__ == "__main__":
    pdf = scrape_tables_to_csv(TARGET_URL)
    pdf.info()

Fetching: https://www.klsescreener.com/v2/screener/quote_results
Found 1 table(s)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Name         1128 non-null   object 
 1   Code         1128 non-null   object 
 2   Category     1128 non-null   object 
 3   Price        1128 non-null   float64
 4   Change       1128 non-null   float64
 5   Change%      1128 non-null   object 
 6   52week       1128 non-null   object 
 7   Volume       1128 non-null   int64  
 8   EPS          1124 non-null   float64
 9   DPS          1128 non-null   float64
 10  NTA          1124 non-null   float64
 11  PE           1124 non-null   float64
 12  DY           1128 non-null   float64
 13  ROE          1128 non-null   float64
 14  PTBV         1120 non-null   float64
 15  MCap.(M)     1128 non-null   float64
 16  Indicators   860 non-null    object 
 17  Unnamed:

In [60]:
import requests
import boto3
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import glob
from datetime import datetime as dt

# ── Session ───────────────────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("MyBursaScraperAPIApp") \
    .getOrCreate()

print("Spark version:", spark.version)

sdf = spark.createDataFrame(pdf)

def clean_column_names(df):
    def normalize(name):
        name = name.replace("%", "Percent")
        name = re.sub(r'[^0-9a-zA-Z_]+', '_', name)
        name = re.sub(r'_+', '_', name)
        return name.strip('_')

    return df.toDF(*[normalize(c) for c in df.columns])

sdf = clean_column_names(sdf)

# ── Transform ─────────────────────────────────────────────────────────────────
sdf = sdf.withColumn(
    "created_at", F.current_timestamp()
)

sdf = sdf.withColumn("year",  F.year("created_at")) \
       .withColumn("month", F.month("created_at")) \
       .withColumn("day", F.day("created_at"))

sdf = sdf.withColumnRenamed("Unnamed_17","OtherInfo")

sdf.printSchema()

# ── Write Parquet to dedicated subfolder ─────────────────────────────────────
local_dir  = "/tmp/klsescreener"                          # subfolder, not root /tmp
#local_path = os.path.join(local_dir, "earthquake_data.parquet")

os.makedirs(local_dir, exist_ok=True)                   # create if not exists

sdf.repartition("year", "month","day") \
  .write \
  .mode("overwrite") \
  .partitionBy("year", "month","day") \
  .option("compression", "snappy") \
  .parquet(local_dir)

# ── Rename part files to deterministic names per partition ────────────────────
parquet_files = []  # list of (local_path, s3_key) tuples

partition_dirs = glob.glob(os.path.join(local_dir, "year=*/month=*/day=*/"))

for partition_dir in sorted(partition_dirs):
    part_files = sorted(glob.glob(os.path.join(partition_dir, "part-*.parquet")))

    for idx, old_path in enumerate(part_files):
        # e.g. data_part_000.parquet, data_part_001.parquet, ...
        fixed_name  = f"data_part_{idx:03d}.parquet"
        fixed_path  = os.path.join(partition_dir, fixed_name)
        os.rename(old_path, fixed_path)

        # Preserve partition folder structure for S3 key
        relative    = os.path.relpath(fixed_path, local_dir)
        s3_key      = f"klsescreener/{relative}"

        parquet_files.append((fixed_path, s3_key))


# ── Find all .parquet part files across all year/month partitions ─────────────
parquet_files = glob.glob(os.path.join(local_dir, "**/*.parquet"), recursive=True)

# ── Upload to MinIO via boto3 ─────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

try:
    for full_local_path in parquet_files:
        # Preserve partition folder structure: year=2023/month=1/part-xxx.parquet
        relative_path = os.path.relpath(full_local_path, local_dir)
        s3_key = f"klsescreener/{relative_path}"

        s3.upload_file(
            Filename=full_local_path,
            Bucket="rawdatasets",
            Key=s3_key
        )
        print(f"Uploaded: {s3_key}")

    print(f"All {len(parquet_files)} file(s) uploaded to MinIO successfully!")
finally:
    # ── Cleanup: only delete /tmp/earthquake subfolder ────────────────────────
    shutil.rmtree(local_dir, ignore_errors=True)
    print(f"Deleted temp directory: {local_dir}")

Spark version: 4.0.2
root
 |-- Name: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Change: double (nullable = true)
 |-- ChangePercent: string (nullable = true)
 |-- 52week: string (nullable = true)
 |-- Volume: long (nullable = true)
 |-- EPS: double (nullable = true)
 |-- DPS: double (nullable = true)
 |-- NTA: double (nullable = true)
 |-- PE: double (nullable = true)
 |-- DY: double (nullable = true)
 |-- ROE: double (nullable = true)
 |-- PTBV: double (nullable = true)
 |-- MCap_M: double (nullable = true)
 |-- Indicators: string (nullable = true)
 |-- OtherInfo: double (nullable = true)
 |-- created_at: timestamp (nullable = false)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- day: integer (nullable = false)

Uploaded: klsescreener/year=2026/month=3/day=21/data_part_000.parquet
All 1 file(s) uploaded to MinIO successfully!
Deleted temp directory: /t

In [ ]:
spark.stop()